<a href="https://colab.research.google.com/github/Shreya08-cyber/CSA6101-digital-forensics/blob/main/data_recovery.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Aim:**

To simulate data recovery (file carving) from a raw storage dump by scanning for specific file header and footer signatures. This technique extracts lost or deleted files without relying on the operating system's file allocation tables.

**Algorithm:**

Start the program.

Define the exact header (Start of File) and footer (End of File) byte sequences for a specific file type (e.g., JPEG).

Create a simulated raw binary disk image containing random noise with a hidden "file" injected in the middle.

Open the raw image file in binary read mode (rb) and read all data into memory.

Scan the byte stream to find the index of the header signature.

Once the header is located, scan forward to locate the corresponding footer signature.

Slice the bytes between (and including) the header and footer to extract the carved file.

Save the extracted byte slice to a new output file and print a success message.

In [2]:
def carve_file(disk_image, header, footer, output_name):
    with open(disk_image, "rb") as f:
        raw_data = f.read()

    start_idx = raw_data.find(header)
    if start_idx == -1:
        print("Header not found. No file carved.")
        return


    end_idx = raw_data.find(footer, start_idx)
    if end_idx == -1:
        print("Footer not found. File is incomplete/corrupted.")
        return


    end_idx += len(footer)
    carved_data = raw_data[start_idx:end_idx]

    with open(output_name, "wb") as out_file:
        out_file.write(carved_data)

    print(f"Successfully carved file between offsets {start_idx} and {end_idx}.")
    print(f"Saved recovered file as: {output_name}")


dummy_disk = "raw_disk.img"
jpeg_header = b'\xFF\xD8\xFF\xE0'
jpeg_footer = b'\xFF\xD9'
hidden_content = b'---THIS IS A RECOVERED DELETED IMAGE---'

with open(dummy_disk, "wb") as f:
    f.write(b'GARBAGEBYTES849302' + jpeg_header + hidden_content + jpeg_footer + b'MOREGARBAGE0923')


carve_file(dummy_disk, jpeg_header, jpeg_footer, "recovered_file.jpg")


with open("recovered_file.jpg", "rb") as f:
    print(f"Recovered Content: {f.read()}")

Successfully carved file between offsets 18 and 63.
Saved recovered file as: recovered_file.jpg
Recovered Content: b'\xff\xd8\xff\xe0---THIS IS A RECOVERED DELETED IMAGE---\xff\xd9'


**Result:**

The algorithm efficiently navigates the unallocated, raw byte stream to find predefined boundaries. It successfully bypasses the need for a file system, excises the hidden data segment, and saves it as a functional, reconstructed file.